# Sunday Project - Data Quality, Insights and CSV Operations

This notebook has Part A and Part B. I used the file sales_raw.csv. All files are in the same folder, so I used relative paths.

## Part A - Data Quality and Basic Insights

Goal: turn the raw sales file into cleaned data, one grouped summary, and two charts.

### 1. Load and inspect

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("sales_raw.csv")
df.head()

In [ ]:
df.shape
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print(df.columns.tolist())
print(df.dtypes)

In [ ]:
# Check missing values and duplicate rows
print("Missing values:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

### 2. Clean the dataset

My cleaning rules:
- Trim extra spaces and make region and category title case, so " food ", "books" and "SOUTH" become Food, Books and South.
- Change sale_date, quantity and unit_price to the right type. Wrong values like "invalid" become missing.
- Drop duplicate rows.
- Drop rows with a missing region, date, quantity or price. I dropped them instead of filling them, because I cannot guess a price or a quantity.
- Drop rows with quantity 0 or less, price 0 or less, or price over 1000. The price 10000 for Stationery is far above all the other prices (40 to 320), so I treated it as a wrong price.
- The original sales_raw.csv file is not changed.

In [ ]:
rows_before = len(df)

# Trim spaces and fix the text in region and category
df["region"] = df["region"].str.strip().str.title()
df["category"] = df["category"].str.strip().str.title()

# Change the columns to the right type
df["sale_date"] = pd.to_datetime(df["sale_date"], errors="coerce")
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")

# Drop duplicate rows
df = df.drop_duplicates()
print("Rows after dropping duplicates:", len(df))

# Drop rows with missing values
df = df.dropna(subset=["region", "sale_date", "quantity", "unit_price"])
print("Rows after dropping missing values:", len(df))

# Drop rows with a wrong quantity or price
df = df[(df["quantity"] > 0) & (df["unit_price"] > 0) & (df["unit_price"] <= 1000)]
df = df.reset_index(drop=True)

rows_after = len(df)
print("Rows before cleaning:", rows_before)
print("Rows after cleaning:", rows_after)
print("Rows removed:", rows_before - rows_after)
df.head()

### 3. Calculate and analyze

In [ ]:
# Add a revenue column: quantity x unit_price
df["revenue"] = df["quantity"] * df["unit_price"]
print("Total revenue:", df["revenue"].sum())
df.head()

In [ ]:
# Summary: revenue by region
revenue_by_region = df.groupby("region")["revenue"].sum().sort_values(ascending=False)
print(revenue_by_region)

# Summary: revenue by category
revenue_by_category = df.groupby("category")["revenue"].sum().sort_values(ascending=False)
print(revenue_by_category)

In [ ]:
# Chart 1: revenue by region
plt.figure(figsize=(8, 5))
revenue_by_region.plot(kind="bar")
plt.title("Revenue by Region")
plt.xlabel("Region")
plt.ylabel("Revenue")
plt.tight_layout()
plt.savefig("revenue_by_region.png")
plt.show()

In [ ]:
# Chart 2: revenue by category
plt.figure(figsize=(8, 5))
revenue_by_category.plot(kind="bar")
plt.title("Revenue by Category")
plt.xlabel("Category")
plt.ylabel("Revenue")
plt.tight_layout()
plt.savefig("revenue_by_category.png")
plt.show()

### Findings

1. North had the highest revenue, 1470, out of total revenue 3775. Food also had the highest revenue by category, 1945, so most of the sales money came from Food, and a lot of it was in the North.
2. Stationery had the lowest revenue by category, 460, and West had the lowest revenue by region, 530, with only 2 cleaned sales. So Stationery and the West region need more sales, or more products, compared to Food and the North.

### 4. Save the result

In [ ]:
df.to_csv("cleaned_sales.csv", index=False)
print("Saved cleaned_sales.csv")
print("Rows:", len(df))
print(df.columns.tolist())

## Part B - CSV File Handling and Manipulation

Goal: edit the cleaned file from Part A and save a separate file called sales_updated.csv.

### 1. Read and add a column

In [ ]:
df2 = pd.read_csv("cleaned_sales.csv")

# Add tax_amount, 5% of revenue
df2["tax_amount"] = df2["revenue"] * 0.05
df2.head()

### 2. Update and remove

In [ ]:
# Increase the unit_price of Books items by 10% and recalculate revenue and tax
df2.loc[df2["category"] == "Books", "unit_price"] = df2.loc[df2["category"] == "Books", "unit_price"] * 1.10
df2["revenue"] = df2["quantity"] * df2["unit_price"]
df2["tax_amount"] = df2["revenue"] * 0.05
df2.head()

In [ ]:
# Remove one row, transaction_id 1001
row_to_remove = df2[df2["transaction_id"] == 1001].index
df2 = df2.drop(row_to_remove)
print("Rows after removing transaction_id 1001:", len(df2))

# Make a temporary column, then remove it
df2["review_status"] = "Checked"
print(df2.columns.tolist())
df2 = df2.drop(columns=["review_status"])
print(df2.columns.tolist())

### 3. Save and verify

In [ ]:
df2.to_csv("sales_updated.csv", index=False)

# Read the saved file again to check it
df_check = pd.read_csv("sales_updated.csv")
df_check.head()
print("Final columns:", df_check.columns.tolist())
print("Rows:", len(df_check))